In [ ]:
!pip install pypdf
from pypdf import PdfReader
!pip install pdfplumber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 941.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 94.2 MB/s eta 0:00:00


In [ ]:
import pdfplumber
import os

PDF_DIR = "/content/2026R"

def extract_pages(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=2)
            if text:
                pages.append(text)
    return pages

In [ ]:
import re

def split_paragraphs(pages):
    full_text = "\n".join(pages)
    paras = re.split(r'(?m)^\s*\d+\.\s', full_text)
    return [p.strip() for p in paras if len(p.strip()) > 20]

In [ ]:
CITATION_PATTERNS = [
    r"\(\d{4}\)\s*\d+\s*SCC\s*\d+",        # (2015) 2 SCC 410
    r"\[\d{4}\]\s*\d+\s*SCR\s*\d+",        # [2014] 13 SCR 735
    r"AIR\s*\d{4}\s*SC\s*\d+",             # AIR 1967 SC 1643
    r"\d{4}\s*SCC\s*OnLine\s*SC\s*\d+"     # 2022 SCC OnLine SC 123
]

def extract_citations(text):
    cites = []
    for pattern in CITATION_PATTERNS:
        cites.extend(re.findall(pattern, text))
    return list(set(cites))

In [ ]:
import os
import json

PDF_DIR = "/content/2026R"

dataset = []

for filename in os.listdir(PDF_DIR):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(PDF_DIR, filename)

        pages = extract_pages(pdf_path)
        paragraphs = split_paragraphs(pages)

        for i, para in enumerate(paragraphs):
            cites = extract_citations(para)
            if cites:
                start = max(0, i - 1)
                end = min(len(paragraphs), i + 2)
                context = " ".join(paragraphs[start:end])

                dataset.append({
                    "case_id": filename.replace(".pdf", ""),
                    "context": context,
                    "citations": cites
                })
with open("all_citations_25_cases.jsonl", "w") as f:
    for row in dataset:
        f.write(json.dumps(row) + "\n")

print("Total citation samples across 25 PDFs:", len(dataset))



Total citation samples across 25 PDFs: 163


In [ ]:
from collections import defaultdict

casewise = defaultdict(list)

for row in dataset:
    casewise[row["case_id"]].append(row)

for case_id, rows in casewise.items():
    print(f"\n===== CASE: {case_id} | Samples: {len(rows)} =====")
    for r in rows:
        print(json.dumps(r, indent=2))


===== CASE: 2026_7 | Samples: 12 =====
{
  "case_id": "2026_7",
  "context": "[2026] 1 S.C.R. 91 : 2026 INSC 7\nNirbhay Singh Suliya\nv.\nState of Madhya Pradesh & Anr.\n(Civil Appeal No. 40 of 2026)\n05 January 2026\n[J.B. Pardiwala* and K.V. Vishwanathan,* JJ.]\nIssue for Consideration\nWhether on facts, based on the four judicial orders of grant of bail\nper se and without anything more, the authorities were justified in\nremoving the appellant-Judicial Officer from service.\nHeadnotes\u2020\nJudiciary \u2013 District judiciary \u2013 Departmental proceedings\nagainst Judicial Officer on the allegation that extraneous\nconsiderations actuated passing of bail orders \u2013 Mere wrong\norder or wrong exercise of discretion in grant of bail by itself\nwithout anything more, not a ground to initiate departmental\nproceedings against Judicial Officers \u2013 Complainant lodged\ncomplaint with the Chief Justice of the High Court inter alia\nalleging that the appellant-Additional District